# Construct GSS dataset restricted to job value respondents

Starts from the constructed GSS dataset (`gss.csv`) and drops every survey response that is missing **all** of the job value variables.

Job value variables come in three nested definitions:
- **core**: questions about how important specific job features are (job security, high income, advancement, etc.)
- **broad**: core + why people work / how meaningful work is (JOBMEANS, WRKEARN, ...)
- **wide**: broad + job preference and trade-off questions (LOWPAY, EMPSELF, ...)

Change `DEFINITION` below to switch.

In [1]:
import pandas as pd

data_dir = "../../data/"
constructed_dir = data_dir + "constructed_datasets/"

DEFINITION = "core"  # "core", "broad", or "wide"

In [2]:
df = pd.read_csv(constructed_dir + "gss.csv", low_memory=False)
df.shape

(75699, 61)

In [4]:
# Importance of / preference for specific job features
CORE_JOB_VALUES = [
    "JOBKEEP", "JOBRISE", "JOBHONOR", "JOBOFF", "JOBINTER", "JOBINDEP",
    "JOBRESP", "JOBPEOP", "JOBHELP", "JOBSOC", "JOBACCMP", "JOBSAFE",
    "IMPPROMO", "SECJOB", "HIINC", "PROMOTN", "LEISURE", "INTJOB",
    "WRKINDP", "HLPOTHS", "HLPSOC", "FLEXHRS", "FLEXHRS1", "WKPERSNL",
]

# Why people work / how meaningful work is
BROAD_EXTRA = ["JOBMEANS", "ILIKEJOB", "WRKEARN", "WRKENJOY", "TIMEPDWK"]

# Job preferences and trade-offs
WIDE_EXTRA = [
    "EMPSELF", "SMALLBIG", "PRIVGOVT", "HRSMONEY", "WANTJOB1", "FAMORJOB",
    "JOBORFAM", "NEWSKILL", "LOWPAY", "TEMPWORK", "MORTRAVL",
]

JOB_VALUE_SETS = {
    "core": CORE_JOB_VALUES,
    "broad": CORE_JOB_VALUES + BROAD_EXTRA,
    "wide": CORE_JOB_VALUES + BROAD_EXTRA + WIDE_EXTRA,
}

job_value_cols = JOB_VALUE_SETS[DEFINITION]
print(f"{DEFINITION}: {len(job_value_cols)} job value variables")

core: 24 job value variables


In [5]:
# How many respondents answered each job value variable?
df[job_value_cols].notna().sum().sort_values(ascending=False)

SECJOB      5579
INTJOB      5563
PROMOTN     5557
HLPOTHS     5556
HLPSOC      5552
HIINC       5552
WRKINDP     5549
FLEXHRS     4069
WKPERSNL    1468
FLEXHRS1    1464
LEISURE     1403
JOBKEEP      918
JOBHELP      918
JOBRISE      915
JOBINTER     915
JOBOFF       915
JOBSAFE      914
JOBACCMP     914
JOBSOC       914
JOBPEOP      914
JOBINDEP     914
JOBHONOR     914
JOBRESP      912
IMPPROMO     899
dtype: int64

In [6]:
# Keep a response if it answered at least one job value variable
has_job_value = df[job_value_cols].notna().any(axis=1)
df_job_values = df[has_job_value].reset_index(drop=True)

print(f"Original:  {df.shape}")
print(f"Dropped:   {(~has_job_value).sum()} responses missing all {DEFINITION} job value variables")
print(f"Final:     {df_job_values.shape}")

Original:  (75699, 61)
Dropped:   68277 responses missing all core job value variables
Final:     (7422, 61)


In [12]:
# stats of category
print("Survey years:", sorted(df_job_values["YEAR"].unique().astype(int)))
print("Answered per response (job value vars):")
print(df_job_values[job_value_cols].notna().sum(axis=1).describe())

satisfaction_cols = ["SATJOB", "JOBSAT", "SATJOBHV"]
has_sat = df_job_values[satisfaction_cols].notna().any(axis=1)
has_onet = df_job_values["ONET_SOC_CODE"].notna()
print(f"With a job satisfaction answer: {has_sat.sum()}")
print(f"With an O*NET code:             {has_onet.sum()}")
print(f"With both:                      {(has_sat & has_onet).sum()}")

Survey years: [np.int64(1982), np.int64(1989), np.int64(1991), np.int64(1998), np.int64(2006), np.int64(2016)]
Answered per response (job value vars):
count    7422.000000
mean        7.974670
std         2.905817
min         1.000000
25%         8.000000
50%         9.000000
75%         9.000000
max        12.000000
dtype: float64
With a job satisfaction answer: 5899
With an O*NET code:             7029
With both:                      5705


In [10]:
# Write to CSV
output_path = constructed_dir + f"gss_job_values_{DEFINITION}.csv"
df_job_values.to_csv(output_path, index=False)
output_path

'../../data/constructed_datasets/gss_job_values_core.csv'

Core is a small set where nearly everyone answered many job value questions (about 8 of the 24 on average). Broad is a much larger set where most people answered just one question. Wide is the same as Broad with 6 extra rows.

Core: "How important is job security to you?" and "How important is high income?" Only people from 6 survey years were asked these, so the file is small.

Broad: adds questions like "Is your job just a way to earn money?" and, above all, JOBMEANS ("work important and feel accomplishment"). JOBMEANS was asked in 20 survey years, so 19,000+ more respondents come in. Most of them answered only that one question.

Wide: adds "Would you accept lower pay?", "Do you prefer self-employment?" and similar. Almost everyone who answered those had already answered a broad question, so only 6 new rows come in.
